In [1]:
import krippendorff
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from itertools import combinations
from sklearn.metrics import cohen_kappa_score
from pathlib import Path
import pandas as pd

# Read Labelled Veteran Data (Excel)

This notebook loads `Labelled-consolidatedVeteranData-May.xlsx` and previews each sheet.

In [ ]:
DATA_PATH = Path("Labelled-consolidatedVeteranData-May.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected Excel file at {DATA_PATH.resolve()}")

In [ ]:
xl = pd.ExcelFile(DATA_PATH)
print("Sheets:", xl.sheet_names)

In [ ]:
sheets = {name: pd.read_excel(xl, sheet_name=name) for name in xl.sheet_names}

for name, df in sheets.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")

## Form Responses 1

Wide form-response export (one column per survey question).

In [ ]:
form_responses = sheets["Form Responses 1"]
print(f"Shape: {form_responses.shape}")
form_responses.iloc[:, :5].head()

## Reformat Form Responses 1

Pivot the wide Google Form export so that:

- **First column** = tweet text (~149 rows)
- **Column headers** = 8 reviewer email addresses (Excel rows 3–10)
- **Values** = each reviewer's label for that tweet

In [ ]:
META_COLS = ["Timestamp", "Email Address", "Score"]

# Excel rows 3-10 (0-based iloc 1:9) are the 8 reviewers
reviewers = form_responses.iloc[1:9].copy()

tweet_cols = [c for c in form_responses.columns if c not in META_COLS]
labeled_tweet_cols = [c for c in tweet_cols if reviewers[c].notna().any()]

print(f"Reviewers: {len(reviewers)}")
print(f"Reviewed tweets: {len(labeled_tweet_cols)}")

reviews_by_tweet = (
    reviewers.set_index("Email Address")[labeled_tweet_cols]
    .T
    .reset_index(names="tweet")
)

reviews_by_tweet.shape

In [ ]:
# First column = tweet, remaining columns = reviewer labels
reviews_by_tweet.head()

## Anonymize reviewer emails

Map each reviewer email to `reviewer1`–`reviewer8` and export a shareable dataset.

In [ ]:
reviewer_emails = [c for c in reviews_by_tweet.columns if c != "tweet"]
reviewer_map = {email: f"reviewer{i + 1}" for i, email in enumerate(reviewer_emails)}

reviewer_id_map = pd.DataFrame(
    {"reviewer_id": reviewer_map.values(), "email": reviewer_map.keys()}
)

veteran_tweet_labels = reviews_by_tweet.rename(columns=reviewer_map)

ANON_DATA_PATH = Path("Veteran_Tweet_Label_PTSD.csv")
MAP_PATH = Path("reviewer_id_map.csv")

veteran_tweet_labels.to_csv(ANON_DATA_PATH, index=False)
reviewer_id_map.to_csv(MAP_PATH, index=False)

print(f"Saved anonymized data to {ANON_DATA_PATH.resolve()}")
print(f"Saved reviewer mapping to {MAP_PATH.resolve()}")
reviewer_id_map

In [ ]:
OUTPUT_PATH = Path("form_responses_reformatted.csv")
reviews_by_tweet.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH.resolve()}")

In [1]:
import pandas as pd
from collections import Counter

# ============================================================
# Read original data
# ============================================================

df = pd.read_csv("Veteran_Tweet_Label_PTSD.csv")

# ============================================================
# Identify reviewer columns
# ============================================================

reviewer_cols = [
    c for c in df.columns
    if c.startswith("reviewer")
]

# ============================================================
# Majority vote
# ============================================================

def majority_vote(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return None

    counts = Counter(labels)
    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # Clear majority
    if len(winners) == 1:
        return winners[0]

    # Tie
    return " | ".join(sorted(winners))


# ============================================================
# Number of votes for majority label
# ============================================================

def majority_vote_count(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return 0

    return max(Counter(labels).values())


# ============================================================
# Create ground-truth columns
# ============================================================

df["ground_truth_label"] = (
    df[reviewer_cols]
    .apply(majority_vote, axis=1)
)

df["ground_truth_votes"] = (
    df[reviewer_cols]
    .apply(majority_vote_count, axis=1)
)

df["ground_truth_agreement"] = (
    df["ground_truth_votes"] / len(reviewer_cols)
)

# ============================================================
# Save
# ============================================================

output_file = "Veteran_Tweet_Label_PTSD_ground_truth.csv"

df.to_csv(output_file, index=False)

# ============================================================
# Results
# ============================================================

print("Ground-truth file created:")
print(output_file)

print("\nGround-truth label distribution:")
print(df["ground_truth_label"].value_counts())

print("\nMajority vote distribution:")
print(df["ground_truth_votes"].value_counts().sort_index())

Ground-truth file created:
Veteran_Tweet_Label_PTSD_ground_truth.csv

Ground-truth label distribution:
ground_truth_label
1-No Concern                                                                                                                      98
2- Distressed Self-Disclosure                                                                                                     17
3-Self-Identified Crisis Symptoms and Seeking Support                                                                             14
6- Not Relevant to the Research                                                                                                    9
1-No Concern | 2- Distressed Self-Disclosure                                                                                       2
5- Acute Crisis with Suicidal Ideation                                                                                             2
2- Distressed Self-Disclosure | 5- Acute Crisis with Suicidal Ideation          

In [2]:
import pandas as pd
from collections import Counter

# ============================================================
# Read merged data
# ============================================================

df = pd.read_csv(
    "Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged.csv"
)

# ============================================================
# Identify reviewer columns
# ============================================================

reviewer_cols = [
    c for c in df.columns
    if c.startswith("reviewer")
]

# ============================================================
# Majority vote
# ============================================================

def majority_vote(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return None

    counts = Counter(labels)
    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # Clear majority
    if len(winners) == 1:
        return winners[0]

    # Tie
    return " | ".join(sorted(winners))


# ============================================================
# Number of votes for majority label
# ============================================================

def majority_vote_count(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return 0

    return max(Counter(labels).values())


# ============================================================
# Create ground-truth columns
# ============================================================

df["ground_truth_label"] = (
    df[reviewer_cols]
    .apply(majority_vote, axis=1)
)

df["ground_truth_votes"] = (
    df[reviewer_cols]
    .apply(majority_vote_count, axis=1)
)

df["ground_truth_agreement"] = (
    df["ground_truth_votes"] / len(reviewer_cols)
)

# ============================================================
# Save
# ============================================================

output_file = (
    "Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged_ground_truth.csv"
)

df.to_csv(output_file, index=False)

# ============================================================
# Results
# ============================================================

print("Ground-truth file created:")
print(output_file)

print("\nGround-truth label distribution:")
print(df["ground_truth_label"].value_counts())

print("\nMajority vote distribution:")
print(df["ground_truth_votes"].value_counts().sort_index())

Ground-truth file created:
Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged_ground_truth.csv

Ground-truth label distribution:
ground_truth_label
1-No Concern                                                                                                                      92
2- Distressed Self-Disclosure                                                                                                     16
3-Self-Identified Crisis Symptoms and Seeking Support                                                                             14
Other                                                                                                                              9
1-No Concern | Other                                                                                                               8
1-No Concern | 2- Distressed Self-Disclosure                                                                                       2
5- Acute Crisis with Suicidal Ideation                